# Notebook 4: Homology (Comparative) Modelling
**ProteinIQ Project**

## Background
Homology modelling predicts a protein's 3D structure by:
1. **Template search** — find evolutionarily related proteins with known structures
2. **Sequence alignment** — align query sequence to templates (Smith-Waterman)
3. **Coordinate transfer** — copy backbone atoms from aligned template residues
4. **Loop modelling** — fill gaps with energy-minimised geometry
5. **Model validation** — Ramachandran plot, Z-score, clash analysis

**Key rule of thumb**: identity ≥30% → reliable model; 20–30% → moderate; <20% → twilight zone.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'backend'))
from models.homology_model import (smith_waterman, blosum62, search_templates,
                                    build_homology_model, TEMPLATE_LIBRARY)
np.random.seed(42)

## 1. BLOSUM62 Substitution Matrix Visualisation

In [ ]:
AA = list('ACDEFGHIKLMNPQRSTVWY')
mat = np.array([[blosum62(a,b) for b in AA] for a in AA])

fig, ax = plt.subplots(figsize=(9,8))
cmap = LinearSegmentedColormap.from_list('bl62',['#A32D2D','#888780','#1D9E75'])
im = ax.imshow(mat, cmap=cmap, vmin=-4, vmax=11)
ax.set_xticks(range(20)); ax.set_xticklabels(AA, fontsize=9)
ax.set_yticks(range(20)); ax.set_yticklabels(AA, fontsize=9)
ax.set_title('BLOSUM62 substitution matrix\n(used in Smith-Waterman alignment)', fontsize=11)
for i in range(20):
    for j in range(20):
        ax.text(j, i, str(mat[i,j]), ha='center', va='center',
                fontsize=7, color='white' if abs(mat[i,j])>3 else 'black')
plt.colorbar(im, ax=ax, label='Substitution score')
plt.tight_layout()
plt.show()

## 2. Smith-Waterman Local Alignment — Step by Step

In [ ]:
query    = 'VLSEGEWQLVLHVWAKVEADVAGHGQDILIR'
template = 'VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEK'

result = smith_waterman(query, template)
print('Smith-Waterman Alignment:')
print(f'  Score    : {result["score"]:.1f}')
print(f'  Identity : {result["identity"]}%')
print(f'  Coverage : {result["coverage"]}%')
print()
print(f'  Query   : {result["aligned_query"]}')
# Midline
mid = ''.join('|' if a==b and a!='-' else ' '
               for a,b in zip(result['aligned_query'],result['aligned_template']))
print(f'           {mid}')
print(f'  Template: {result["aligned_template"]}')

# Visualise alignment
fig, ax = plt.subplots(figsize=(14, 2.5))
aq, at = result['aligned_query'], result['aligned_template']
for i,(qa,ta) in enumerate(zip(aq,at)):
    match = qa == ta and qa != '-'
    gap   = qa == '-' or ta == '-'
    color = '#1D9E75' if match else ('#D85A30' if gap else '#888780')
    ax.add_patch(plt.Rectangle((i, 0.5), 0.9, 0.8, color=color, alpha=0.7))
    ax.add_patch(plt.Rectangle((i, 0.0), 0.9, 0.4, color=color, alpha=0.5))
    ax.text(i+0.45, 0.9, qa, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')
    ax.text(i+0.45, 0.2, ta, ha='center', va='center', fontsize=9,
            fontweight='bold', color='white')
ax.set_xlim(0, len(aq))
ax.set_ylim(-0.2, 1.5)
ax.set_yticks([0.2, 0.9]); ax.set_yticklabels(['Template', 'Query'])
ax.set_xlabel('Alignment position')
ax.set_title(f'Smith-Waterman alignment — Identity: {result["identity"]}%  Coverage: {result["coverage"]}%')
patch_m = mpatches.Patch(color='#1D9E75', alpha=0.7, label='Match')
patch_mm= mpatches.Patch(color='#888780', alpha=0.7, label='Mismatch')
patch_g = mpatches.Patch(color='#D85A30', alpha=0.7, label='Gap')
ax.legend(handles=[patch_m,patch_mm,patch_g], loc='upper right')
plt.tight_layout()
plt.show()

## 3. Template Search Against the Library

In [ ]:
query_seq = 'VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEK'
print(f'Query: {query_seq[:40]}...')
print(f'Length: {len(query_seq)} residues\n')

hits = search_templates(query_seq, top_k=5)
print(f'{"Rank":<5}{"ID":<8}{"Name":<30}{"Identity":<12}{"Coverage":<12}{"Score"}')
print('-'*75)
for i, h in enumerate(hits):
    print(f'{i+1:<5}{h["template_id"]:<8}{h["template_name"]:<30}{h["identity"]:>6.1f}%    '
          f'{h["coverage"]:>6.1f}%    {h["score"]:>8.1f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = [h['template_name'][:15] for h in hits[:5]]
axes[0].barh(names[::-1], [h['score'] for h in hits[:5]][::-1], color='#1D9E75', alpha=0.8)
axes[0].set_xlabel('SW Score'); axes[0].set_title('Template ranking by alignment score')
axes[0].grid(axis='x', alpha=0.3)
axes[1].barh(names[::-1], [h['identity'] for h in hits[:5]][::-1], color='#378ADD', alpha=0.8)
axes[1].axvline(30, color='#D85A30', ls='--', lw=1.5, label='30% threshold')
axes[1].set_xlabel('Sequence identity (%)')
axes[1].set_title('Sequence identity of top templates')
axes[1].legend(); axes[1].grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Full Homology Model Build + Validation

In [ ]:
import sys; sys.path.insert(0,'../backend')
from models.ramachandran import validate_structure

model = build_homology_model(query_seq, top_k=3, seed=42)

print('=== Homology Model Report ===')
print(f'Template used : {model.best_template["template_id"]} '
      f'({model.best_template["template_name"]})')
print(f'Seq. identity : {model.seq_identity:.1f}%')
print(f'Coverage      : {model.coverage:.1f}%')
print(f'Model quality : {model.model_quality.upper()}')

# Validate the model
report = validate_structure(model.model_coords, sequence=query_seq)
print(f'\n--- Ramachandran Validation ---')
print(f'Grade         : {report.grade}')
print(f'Core region   : {report.pct_core:.1f}%')
print(f'Outliers      : {report.pct_outliers:.1f}%')
print(f'Z-score       : {report.z_score}')

# Visualise Ramachandran
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ss_colors = {'H':'#D85A30','E':'#378ADD','C':'#888780'}
for r in report.per_residue:
    if r['phi'] is not None and r['psi'] is not None:
        axes[0].scatter(r['phi'], r['psi'],
                        color=ss_colors.get(r['ss'],'#888'),
                        alpha=0.7, s=40, edgecolors='none')
axes[0].axvline(0, color='gray', lw=0.5, alpha=0.4)
axes[0].axhline(0, color='gray', lw=0.5, alpha=0.4)
axes[0].set_xlabel('φ (degrees)'); axes[0].set_ylabel('ψ (degrees)')
axes[0].set_xlim(-180,180); axes[0].set_ylim(-180,180)
axes[0].set_title(f'Ramachandran plot — Grade {report.grade}\n'
                   f'Core: {report.pct_core:.0f}%  Outliers: {report.pct_outliers:.0f}%')
for lbl,c in ss_colors.items():
    axes[0].scatter([],[], color=c, label=lbl)
axes[0].legend(title='SS', fontsize=9)

# 3D Cα trace
ax3d = fig.add_subplot(122, projection='3d') if hasattr(plt,'cm') else axes[1]
try:
    from mpl_toolkits.mplot3d import Axes3D
    ax3d = fig.add_subplot(122, projection='3d')
    cs = model.model_coords
    ax3d.plot(cs[:,0], cs[:,1], cs[:,2], 'k-', lw=0.8, alpha=0.4)
    for r in report.per_residue:
        i=r['residue']
        if i<len(cs):
            c=ss_colors.get(r['ss'],'#888')
            ax3d.scatter(cs[i,0],cs[i,1],cs[i,2],color=c,s=30,alpha=0.8,depthshade=True)
    ax3d.set_title('Homology model Cα trace')
    ax3d.set_axis_off()
except:
    axes[1].text(0.5,0.5,'3D plot requires mpl_toolkits',ha='center',va='center',transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('../data/figures/homology_model.png', dpi=150, bbox_inches='tight')
plt.show()